# Algorithm 8.2 — Interplanetary trajectory (the capstone)

**Goal:** design a transfer from one planet to another. Given a **departure** (planet + date) and an **arrival** (planet + date), find the heliocentric transfer orbit and the spacecraft velocities at both ends — and hence the hyperbolic excess velocities the rockets must provide.

**Code (answer key):** [`interplanetary`](../curtis/algorithms/alg_8_2_interplanetary.py) · **Book:** §8.10, Algorithm 8.2 · **Worked example:** 8.8

This is the book's grand finale, and it is almost pure orchestration: **8.1** locates the two planets, **5.2 (Lambert)** connects them in the given flight time.

## Read first

| Symbol | Meaning |
|---|---|
| `depart` | `[planet_id, year, month, day, hour, minute, second]` at departure |
| `arrive` | same, at arrival |
| $\mathbf{R}_{p1},\mathbf{V}_{p1}$ | departure planet's heliocentric state (from 8.1) |
| $\mathbf{R}_{p2},\mathbf{V}_{p2}$ | arrival planet's state |
| $\mathbf{V}_1,\mathbf{V}_2$ | spacecraft heliocentric velocities at departure/arrival (from Lambert) |
| $\mathbf{v}_\infty=\mathbf{V}-\mathbf{V}_p$ | hyperbolic excess velocity (what the rocket must supply / arrives with) |

Frame: **heliocentric ecliptic**, $\mu=\mu_\odot$.

## The picture (patched conics)

A real interplanetary trip is three conics stitched together:

1. **Departure hyperbola** — escape from planet 1's gravity well.
2. **Heliocentric transfer ellipse** — coasting under the Sun alone from planet 1 to planet 2.
3. **Arrival hyperbola** — capture (or fly-by) at planet 2.

Each planet's sphere of influence is tiny next to the Sun–planet distance, so we **patch** the conics: treat the spacecraft as leaving from planet 1's *position* and arriving at planet 2's *position*, and let the Sun govern the long middle leg.

Algorithm 8.2 solves that middle leg: it is a **Lambert problem** between the two planet positions over the flight time. The spacecraft's heliocentric departure velocity $\mathbf{V}_1$ differs from the planet's $\mathbf{V}_{p1}$ by the **hyperbolic excess** $\mathbf{v}_{\infty 1}=\mathbf{V}_1-\mathbf{V}_{p1}$ — exactly the velocity the launch must add on top of riding along with the planet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "..")
# Prerequisites you already built:
from curtis.algorithms.alg_8_1_planet_elements_and_sv import planet_elements_and_sv
from curtis.algorithms.alg_5_2_lambert import lambert
from curtis.algorithms.alg_4_1_coe_from_sv import coe_from_sv
from curtis.algorithms.alg_4_2_sv_from_coe import sv_from_coe
# Reference, for the picture only:
from curtis.algorithms.alg_8_2_interplanetary import interplanetary as _itp_ref

In [ ]:
# Example 8.8: Earth (1996-11-07) -> Mars (1997-09-12), heliocentric transfer.
mu_sun = 1.327124e11
AU = 149597871.0
deg = np.pi/180
depart = [3, 1996, 11, 7, 0, 0, 0]
arrive = [4, 1997, 9, 12, 0, 0, 0]

(Rp1, Vp1, jd1), (Rp2, Vp2, jd2), (V1, V2) = _itp_ref(depart, arrive, mu_sun)

def orbit_xy(spec):
    pid = spec[0]
    coe = planet_elements_and_sv(pid, *spec[1:], mu_sun)[0]   # spec[1:] = date/time
    h, e, RA, incl, w = coe[0], coe[1], coe[2], coe[3], coe[4]
    th = np.linspace(0, 2*np.pi, 400)
    P = np.array([sv_from_coe([h, e, RA*deg, incl*deg, w*deg, t], mu_sun)[0] for t in th])
    return P[:, 0]/AU, P[:, 1]/AU

# transfer arc from Earth-at-departure to Mars-at-arrival
hT, eT, RAT, iT, wT, TA1, aT = coe_from_sv(Rp1, V1, mu_sun)
TA2 = coe_from_sv(Rp2, V2, mu_sun)[5]
ta2 = TA2 + (2*np.pi if TA2 < TA1 else 0)
arc = np.linspace(TA1, ta2, 300)
T = np.array([sv_from_coe([hT, eT, RAT, iT, wT, t], mu_sun)[0] for t in arc])

fig, ax = plt.subplots(figsize=(6.8, 6.8))
ax.plot(0, 0, 'o', color='#F2A623', ms=13); ax.annotate('Sun', (0, 0),
        textcoords='offset points', xytext=(8, -12), color='#B05535')
ex, ey = orbit_xy(depart); ax.plot(ex, ey, color='#3B8BD4', lw=1.2, label='Earth orbit')
mx, my = orbit_xy(arrive); ax.plot(mx, my, color='#D4537E', lw=1.2, label='Mars orbit')
ax.plot(T[:, 0]/AU, T[:, 1]/AU, color='#1D9E75', lw=2.2, label='transfer')
ax.plot(Rp1[0]/AU, Rp1[1]/AU, 'o', color='#3B8BD4', ms=8)
ax.plot(Rp2[0]/AU, Rp2[1]/AU, 'o', color='#D4537E', ms=8)
ax.annotate('Earth at departure', (Rp1[0]/AU, Rp1[1]/AU),
            textcoords='offset points', xytext=(8, 4), color='#3B8BD4', fontsize=9)
ax.annotate('Mars at arrival', (Rp2[0]/AU, Rp2[1]/AU),
            textcoords='offset points', xytext=(-8, 8), color='#D4537E', fontsize=9, ha='right')
ax.set_aspect('equal'); ax.axis('off'); ax.legend(loc='lower right', frameon=False)
ax.set_title('Patched-conic transfer, Earth -> Mars (Example 8.8)')
plt.show()

## See it — the transfer ellipse between two moving targets

The green arc is the heliocentric transfer: it leaves **Earth's position on 1996-11-07** and arrives at **Mars's position on 1997-09-12** — note Mars is somewhere else entirely by arrival, which is why launch windows matter. The whole trick of Algorithm 8.2 is aiming not at where Mars *is* but where it *will be* after the 309-day flight, which is precisely what the Lambert solve enforces. The hyperbolic excess velocities $\mathbf{v}_\infty$ are the small differences between the green transfer's end velocities and the planets' own velocities.

## Where these equations come from

### The patched-conic simplification
Rigorously the spacecraft feels Earth, Sun and Mars simultaneously. But a planet's sphere of influence ($\sim10^6$ km) is minute beside the $\sim10^8$ km Sun–planet distances, so to first order:
- inside an SOI, only the planet matters (a hyperbola);
- everywhere else, only the Sun matters (the transfer conic).

"Patching" means handing the spacecraft from one conic to the next at the SOI boundary — and because that boundary is so close to the planet on the heliocentric scale, we approximate $\mathbf{R}_1\approx\mathbf{R}_{p1}$ and $\mathbf{R}_2\approx\mathbf{R}_{p2}$.

### The middle leg is a Lambert problem
Departure and arrival positions come from Algorithm 8.1; the time of flight is $\Delta t=(\text{jd}_2-\text{jd}_1)\times86400$. Feeding $(\mathbf{R}_1,\mathbf{R}_2,\Delta t)$ to Lambert (5.2) gives the transfer-orbit velocities $\mathbf{V}_1,\mathbf{V}_2$.

### Hyperbolic excess velocity
On the transfer, the craft moves at $\mathbf{V}_1$; the planet it just left moves at $\mathbf{V}_{p1}$. The launch only has to supply the **difference**,
$$\mathbf{v}_{\infty1}=\mathbf{V}_1-\mathbf{V}_{p1},\qquad \mathbf{v}_{\infty2}=\mathbf{V}_2-\mathbf{V}_{p2},$$
the hyperbolic excess speeds that size the departure burn and the arrival encounter. (These two lines are done in the verify, not inside `interplanetary` itself.)

## Step by step (in code order)

1. **Departure planet state** (Algorithm 8.1): `_, Rp1, Vp1, jd1 = planet_elements_and_sv(*depart, mu)`.
2. **Arrival planet state:** `_, Rp2, Vp2, jd2 = planet_elements_and_sv(*arrive, mu)`.
3. **Time of flight:** `tof = (jd2 - jd1)*24*3600` seconds.
4. **Patched conic:** `R1 = Rp1`, `R2 = Rp2`.
5. **Lambert (prograde):** `V1, V2 = lambert(R1, R2, tof, "pro", mu)`.
6. **Return** `(Rp1, Vp1, jd1)`, `(Rp2, Vp2, jd2)`, `(V1, V2)`.

**↓ Now type the implementation below.** It is short — the heavy lifting is in `planet_elements_and_sv` and `lambert`, which you have already built. Answer key linked above; peek only after you try.

In [ ]:
def interplanetary(depart, arrive, mu):
    """Patched-conic transfer from one planet to another.

    depart, arrive = [planet_id, year, month, day, hour, minute, second].
    Returns (Rp1, Vp1, jd1), (Rp2, Vp2, jd2), (V1, V2).
    """
    # 1. Departure planet state (Algorithm 8.1):
    #        _, Rp1, Vp1, jd1 = planet_elements_and_sv(*depart, mu)
    # 2. Arrival planet state:
    #        _, Rp2, Vp2, jd2 = planet_elements_and_sv(*arrive, mu)
    # 3. tof = (jd2 - jd1)*24*3600
    # 4. R1 = Rp1;  R2 = Rp2                      # patched-conic assumption
    # 5. V1, V2 = lambert(R1, R2, tof, "pro", mu)
    # return (Rp1, Vp1, jd1), (Rp2, Vp2, jd2), (V1, V2)
    raise NotImplementedError("fill in steps 1-5, then delete this line")

## Verify — Example 8.8

**Input:** depart Earth 1996-11-07 00:00, arrive Mars 1997-09-12 00:00, $\mu_\odot=1.327124\times10^{11}$.

**Expected:** time of flight $=309$ days; $\mathbf{V}_1=[-24.4282,21.7819,0.948049]$, $\mathbf{v}_{\infty1}$ magnitude $=3.16513$ km/s; $\mathbf{V}_2=[22.1581,-0.19666,-0.457847]$, $\mathbf{v}_{\infty2}$ magnitude $=2.88518$ km/s; transfer orbit $e=0.205785$, $a=1.84742\times10^8$ km.

Run once your function is typed.

In [ ]:
mu_sun = 1.327124e11
depart = [3, 1996, 11, 7, 0, 0, 0]
arrive = [4, 1997, 9, 12, 0, 0, 0]

(Rp1, Vp1, jd1), (Rp2, Vp2, jd2), (V1, V2) = interplanetary(depart, arrive, mu_sun)

tof_days = jd2 - jd1
vinf1 = V1 - Vp1
vinf2 = V2 - Vp2
print(f"time of flight = {tof_days:.0f} days   (expect 309)")
print(f"V1 (km/s) = [{V1[0]:.6g} {V1[1]:.6g} {V1[2]:.6g}]")
print(f"|v_inf| at departure = {np.linalg.norm(vinf1):.6g} km/s   (expect 3.16513)")
print(f"V2 (km/s) = [{V2[0]:.6g} {V2[1]:.6g} {V2[2]:.6g}]")
print(f"|v_inf| at arrival   = {np.linalg.norm(vinf2):.6g} km/s   (expect 2.88518)")

assert abs(tof_days - 309) < 0.5
assert np.allclose(V1, [-24.4282, 21.7819, 0.948049], atol=1e-3)
assert np.allclose(V2, [22.1581, -0.19666, -0.457847], atol=1e-3)
assert abs(np.linalg.norm(vinf1) - 3.16513) < 1e-4
assert abs(np.linalg.norm(vinf2) - 2.88518) < 1e-4

deg = np.pi/180
coe = coe_from_sv(Rp1, V1, mu_sun)
print(f"\ntransfer orbit: e = {coe[1]:.6g}  a = {coe[6]:.6g} km")
assert abs(coe[1] - 0.205785) < 1e-5 and abs(coe[6] - 1.84742e8) < 1e4
print("\nAll checks passed ✔")

## What this confirms

- An interplanetary trajectory is **three patched conics**; Algorithm 8.2 solves the heliocentric middle leg as a Lambert problem between two planet positions.
- **Launch windows** are inherent: you target where the destination planet *will be* after the flight time, not where it is now.
- $\mathbf{v}_\infty=\mathbf{V}-\mathbf{V}_p$ is the mission-design payoff — the hyperbolic excess that sizes the departure burn and the arrival encounter.
- It is the synthesis of the whole book: ephemeris (8.1) ← elements/state (4.1/4.2) ← Kepler & universal variables (Ch 3); transfer ← Lambert (5.2).

**This completes the Appendix-D study-notebook series** — every numbered algorithm now has a learn-by-typing notebook, from Kepler's equation through interplanetary mission design. 🛰️